In [1]:
from datasets import load_dataset, Audio, DatasetDict

from pyha_analyzer.preprocessors.birdset_event_mapper import XCEventMapping

from pyha_analyzer.preprocessors.smart_sampling import smart_sampling

from pyha_analyzer.preprocessors.birdset_one_hot import classes_one_hot


In [2]:
birdset_data = load_dataset("DBD-research-group/BirdSet", "HSN", trust_remote_code=True)


In [3]:
sampling_rate = 32_000

In [4]:
birdset_data = birdset_data.cast_column(
  column="audio",
  feature=Audio(
      sampling_rate=sampling_rate,
      mono=True,
      decode=True,
  ),
)

In [5]:
birdset_data = DatasetDict(
    {split: birdset_data[split] for split in ["train", "test_5s"]}
)

event_mapper = XCEventMapping()

In [6]:
print(">> Mapping train data.")
birdset_data["train"] = birdset_data["train"].map(
    event_mapper,
    remove_columns=["audio"],
    batched=True,
    batch_size=300,
    desc="Train event mapping",
)

>> Mapping train data.


In [7]:
birdset_data["train"] = birdset_data["train"].remove_columns(["audio"])

In [8]:
# defaults for HSN BirdSet

class_limit = 500
event_limit = 5 


In [9]:
birdset_data["train"] = smart_sampling(
  dataset=birdset_data["train"],
  label_name="ebird_code",
  class_limit=class_limit,
  event_limit=event_limit,
)

sampling: unique-identifier:   0%|          | 0/38170 [00:00<?, ? examples/s]

reached
                                                filepath start_time end_time  \
0      /home/s.dalal.334/.cache/huggingface/datasets/...       None     None   
1      /home/s.dalal.334/.cache/huggingface/datasets/...       None     None   
2      /home/s.dalal.334/.cache/huggingface/datasets/...       None     None   
3      /home/s.dalal.334/.cache/huggingface/datasets/...       None     None   
4      /home/s.dalal.334/.cache/huggingface/datasets/...       None     None   
...                                                  ...        ...      ...   
38165  /home/s.dalal.334/.cache/huggingface/datasets/...       None     None   
38166  /home/s.dalal.334/.cache/huggingface/datasets/...       None     None   
38167  /home/s.dalal.334/.cache/huggingface/datasets/...       None     None   
38168  /home/s.dalal.334/.cache/huggingface/datasets/...       None     None   
38169  /home/s.dalal.334/.cache/huggingface/datasets/...       None     None   

      low_freq high_freq  ebird

sampling: 100%|██████████| 21/21 [00:01<00:00, 11.40it/s]


In [10]:
num_classes = len(
  birdset_data["train"].features["ebird_code"].names
)


In [11]:
for split in ["train", "test_5s"]:
  birdset_data[split] = birdset_data[split].map(
    classes_one_hot,
    batched=True,
    batch_size=300,
    load_from_cache_file=True,
    desc=f"One-hot-encoding {split} labels.",
    fn_kwargs={"num_classes": 21}
  )


One-hot-encoding train labels.:   0%|          | 0/17940 [00:00<?, ? examples/s]

/home/s.dalal.334/pyha-analyzer-2.0/pyha_analyzer/preprocessors/birdset_one_hot.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  class_one_hot_matrix = torch.tensor(class_one_hot_matrix, dtype=torch.float32)


In [13]:
xc_ds = birdset_data["train"].train_test_split(
  test_size=0.2, stratify_by_column="ebird_code" #still works since not multilabel
)

In [14]:
birdset_data["train"] = xc_ds["train"]
birdset_data["test"] = xc_ds["test"]


In [15]:
birdset_data

DatasetDict({
    train: Dataset({
        features: ['filepath', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'ebird_code_multilabel', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel'],
        num_rows: 14352
    })
    test_5s: Dataset({
        features: ['audio', 'filepath', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'ebird_code_multilabel', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel', 'labels'],
        num_rows: 12000
    })
    test: Dataset({
        features: